# Gemma 4 (2B) Fine-Tuning with KerasHub and LoRA (Colab Pro / H100)

Notebook ported 1:1 from `main_pretrained_gemma.py` to run on Google Colab Pro (H100 / A100) in full precision (`float32`).

## 1. Install Dependencies and Configure Backend

In [ ]:
!pip install -q -U keras-hub keras-nlp keras kagglehub

In [ ]:
import os

os.environ["KERAS_BACKEND"] = "torch"

import json
import keras
from keras import layers, ops
import keras_hub
import numpy as np
import tensorflow as tf

# Prevent TensorFlow (used only for tf.data) from pre-allocating all GPU VRAM
tf.config.set_visible_devices([], "GPU")

print("Keras version:", keras.__version__)
print("KerasHub version:", keras_hub.__version__)
print("Backend:", keras.backend.backend())

## 2. Kaggle Authentication (for KerasHub Presets)

In [ ]:
try:
    from google.colab import userdata
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    print("Loaded Kaggle credentials from Colab Secrets.")
except Exception:
    import kagglehub
    kagglehub.login()

## 3. Load `Gemma4CausalLM` (`float32`) and Test Before Fine-Tuning

In [ ]:
gemma_lm = keras_hub.models.Gemma4CausalLM.from_preset(
    "gemma4_2b",
    dtype="float32",
)

gemma_lm.summary()
# Preprocessor: "gemma4_causal_lm_preprocessor"
#  Total params: 5,104,298,467 (19.01 GB)
#  Trainable params: 5,104,186,912 (19.01 GB)
#  Non-trainable params: 111,555 (435.76 KB)

In [ ]:
gemma_lm.compile(sampler="greedy")
print(gemma_lm.generate("A piece of advice", max_length=40))
print(gemma_lm.generate("How can I make brownies?", max_length=40))

## 4. Prepare Fine-Tuning Dataset (`databricks-dolly-15k`)

In [ ]:
# --- START DATA FOR FINE TUNING
PROMPT_TEMPLATE = "[instruction]\n{}[end]\n[response]\n"
RESPONSE_TEMPLATE = "{}[end]"

dataset_path = keras.utils.get_file(
    origin=(
        "https://hf.co/datasets/databricks/databricks-dolly-15k/"
        "resolve/main/databricks-dolly-15k.jsonl"
    ),
)
data = {"prompts": [], "responses": []}
with open(dataset_path) as file:
    for line in file:
        features = json.loads(line)
        if features["context"]:
            continue
        data["prompts"].append(PROMPT_TEMPLATE.format(features["instruction"]))
        data["responses"].append(RESPONSE_TEMPLATE.format(features["response"]))

# --- START OF DATASET
ds = tf.data.Dataset.from_tensor_slices(data).shuffle(2000).batch(2)
val_ds = ds.take(100)
train_ds = ds.skip(100)

## 5. Enable LoRA, Compile, and Train

In [ ]:
# --- START ENABLE LORA
gemma_lm.backbone.enable_lora(rank=8)
gemma_lm.preprocessor.sequence_length = 512

# --- COMPILE AND TRAIN
gemma_lm.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=keras.optimizers.Adam(5e-5),
    weighted_metrics=[keras.metrics.SparseCategoricalAccuracy()],
)
gemma_lm.fit(train_ds, validation_data=val_ds, epochs=1)

## 6. Test After Fine-Tuning

In [ ]:
# --- START TEST
print(
    gemma_lm.generate(
        "[instruction]\nHello from the mountain[end]\n"
        "[response]\n",
        max_length=512,
    )
)
print(
    gemma_lm.generate(
        "[instruction]\nWhat is going on?[end]\n"
        "[response]\n",
        max_length=512,
    )
)
print(
    gemma_lm.generate(
        "[instruction]\nWho are you?[end]\n"
        "[response]\n",
        max_length=512,
    )
)